In [10]:
WAREHOUSE   = "analytics_warehouse"
SCHEMA      = "gold"
TARGET      = f"{WAREHOUSE}.{SCHEMA}.FactTaxiDaily"

year        = 2024  
write_mode  = "overwrite"

GOLD_BASE = (
    "abfss://itransition_de_project@onelake.dfs.fabric.microsoft.com"
    "/gold.Lakehouse"
)

StatementMeta(, d1a35408-c05a-4115-b556-0c1e5e6bbe21, 12, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, FloatType

spark = SparkSession.builder.getOrCreate()


StatementMeta(, d1a35408-c05a-4115-b556-0c1e5e6bbe21, 4, Finished, Available, Finished, False)

In [3]:
taxi_df = (
    spark.table("taxi_trips")
    .filter(F.col("_year") == year)   # partition pruning
)

raw_count = taxi_df.count()
print(f"Silver taxi rows for {year} : {raw_count:,}")

fx_df = (
    spark.table("fx_daily")
    .withColumn("year", F.year("rate_date").cast(IntegerType()))
    .filter(F.col("year") == year)
    .agg(F.avg("usd_eur_rate").alias("avg_usd_eur_rate"))
)

avg_rate_row = fx_df.collect()[0]
avg_rate     = avg_rate_row["avg_usd_eur_rate"] if avg_rate_row["avg_usd_eur_rate"] else 1.0
print(f"Yearly avg USD/EUR rate for {year} : {avg_rate:.6f}")

# DimZone for airport flag join
zone_df = (
    spark.table("dim_zone")
    .select("location_id", "is_airport")
    .withColumnRenamed("location_id", "pu_location_id")
    .withColumnRenamed("is_airport",  "zone_is_airport")
)

StatementMeta(, d1a35408-c05a-4115-b556-0c1e5e6bbe21, 5, Finished, Available, Finished, False)

Silver taxi rows for 2024 : 35,490,943
Yearly avg USD/EUR rate for 2024 : 1.082380


In [4]:
daily_df = (
    taxi_df
    .withColumn("date_key",
        F.date_format("pickup_date", "yyyyMMdd").cast(IntegerType()))
    .groupBy("date_key", "pu_location_id")
    .agg(
        F.count("*")                             .alias("total_trips"),
        F.sum("passenger_count")                 .alias("total_passengers"),
        F.round(F.sum("fare_amount"),    2)      .alias("sum_fare_usd"),
        F.round(F.avg("fare_amount"),    4)      .alias("avg_fare_usd"),
        F.round(F.sum("total_amount"),   2)      .alias("sum_total_amount_usd"),
        F.round(F.avg("total_amount"),   4)      .alias("avg_total_amount_usd"),
        F.round(F.sum("tip_amount"),     2)      .alias("sum_tip_usd"),
        F.round(F.avg("tip_amount"),     4)      .alias("avg_tip_usd"),
        F.round(F.sum("trip_distance"),  2)      .alias("sum_distance_miles"),
        F.round(F.avg("trip_distance"),  4)      .alias("avg_distance_miles"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
    )
    .withColumn("total_trips",      F.col("total_trips").cast(LongType()))
    .withColumn("total_passengers", F.col("total_passengers").cast(LongType()))
)


StatementMeta(, d1a35408-c05a-4115-b556-0c1e5e6bbe21, 6, Finished, Available, Finished, False)

In [5]:
enriched_df = (
    daily_df
    .join(zone_df, on="pu_location_id", how="left")
    .withColumn("sum_fare_eur",
        F.round(F.col("sum_fare_usd") / F.lit(avg_rate), 2))
    .withColumn("avg_fare_eur",
        F.round(F.col("avg_fare_usd") / F.lit(avg_rate), 4))
    .withColumn("airport_trips",
        F.when(F.col("zone_is_airport") == 1, F.col("total_trips"))
         .otherwise(F.lit(0).cast(LongType())))
    .withColumn("pct_airport",
        F.round(
            F.when(F.col("total_trips") > 0,
                F.col("airport_trips") / F.col("total_trips") * 100)
            .otherwise(0.0),
            2
        )
    )
    .select(
        "date_key", "pu_location_id",
        "total_trips", "total_passengers",
        "sum_fare_usd", "avg_fare_usd",
        "sum_total_amount_usd", "avg_total_amount_usd",
        "sum_fare_eur", "avg_fare_eur",
        "sum_tip_usd", "avg_tip_usd",
        "sum_distance_miles", "avg_distance_miles",
        "avg_duration_min",
        "airport_trips", "pct_airport",
    )
)

StatementMeta(, d1a35408-c05a-4115-b556-0c1e5e6bbe21, 7, Finished, Available, Finished, False)

In [9]:
FACT_TAXI_PATH = f"{GOLD_BASE}/Tables/dbo/facttaxidaily"

fact_count = enriched_df.count()
print(f"\nFact rows (date x zone) : {fact_count:,}")

(
    enriched_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(FACT_TAXI_PATH)
)
print(f"[OK] FactTaxiDaily written to gold lakehouse")

StatementMeta(, d1a35408-c05a-4115-b556-0c1e5e6bbe21, 11, Finished, Available, Finished, False)


Fact rows (date x zone) : 80,244
[OK] FactTaxiDaily written to gold lakehouse


In [11]:
val_df = spark.read.format("delta").load(FACT_TAXI_PATH)

print("\n--- FactTaxiDaily summary ---")
val_df.agg(
    F.min("date_key").alias("earliest_date_key"),
    F.max("date_key").alias("latest_date_key"),
    F.sum("total_trips").alias("grand_total_trips"),
    F.round(F.sum("sum_fare_usd") / 1e6, 2).alias("total_revenue_mm_usd"),
    F.round(F.avg("avg_fare_usd"), 2).alias("overall_avg_fare"),
    F.count("*").alias("fact_rows"),
).show()

print("--- Top 5 pickup zones by total trips ---")
(
    val_df
    .groupBy("pu_location_id")
    .agg(F.sum("total_trips").alias("trips"))
    .orderBy(F.desc("trips"))
    .limit(5)
    .show()
)

StatementMeta(, d1a35408-c05a-4115-b556-0c1e5e6bbe21, 13, Finished, Available, Finished, True)


--- FactTaxiDaily summary ---
+-----------------+---------------+-----------------+--------------------+----------------+---------+
|earliest_date_key|latest_date_key|grand_total_trips|total_revenue_mm_usd|overall_avg_fare|fact_rows|
+-----------------+---------------+-----------------+--------------------+----------------+---------+
|         20240101|       20241231|         35490943|              700.13|           31.19|    80244|
+-----------------+---------------+-----------------+--------------------+----------------+---------+

--- Top 5 pickup zones by total trips ---
+--------------+-------+
|pu_location_id|  trips|
+--------------+-------+
|           132|1835623|
|           237|1762883|
|           161|1733621|
|           236|1553616|
|           162|1307013|
+--------------+-------+

